In [2]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==========================================
# SCRIPT DE PREPARACIÓN PARA POWER BI 📊
# ==========================================

print("👨‍💻1. Cargando datos originales...")
df = pd.read_csv('/content/walmart_ml_ready.csv')

print("🧠 2. Recuperando los Clústers de Machine Learning")
# Calculamos ingresos y agrupamos
df['ingresos'] = df['quantity_sold'] * df['unit_price']
df_clientes = df.groupby('customer_id').agg(
    frecuencia=('customer_id', 'count'),
    gasto_total=('ingresos', 'sum')
).reset_index()

# Escalamos y corremos K-Means
scaler = StandardScaler()
clientes_escalados = scaler.fit_transform(df_clientes[['frecuencia', 'gasto_total']])
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
df_clientes['cluster'] = kmeans.fit_predict(clientes_escalados)

# Bautizamos los segmentos según lo que descubrimos la semana pasada
nombres_segmentos = {
    2: 'Platinum (VIP)',
    3: 'Gold (High Ticket)',
    1: 'Silver (Leales)',
    0: 'Bronze (Casuales)'
}
df_clientes['segmento_ml'] = df_clientes['cluster'].map(nombres_segmentos)

print("🔗 3. Fusionando segmentos con la tabla principal de transacciones...")
df_master = pd.merge(df, df_clientes[['customer_id', 'segmento_ml']], on='customer_id', how='left')

print("⚙️ 4. Creando KPIs logísticos para el Dashboard...")
# Esto nos dirá qué tan cerca está un producto de quedarse en ceros
df_master['inventory_health_ratio'] = (df_master['inventory_level'] / df_master['reorder_point']).round(2)
df_master['alerta_stockout'] = df_master['inventory_health_ratio'].apply(lambda x: 'Crítico' if x <= 1 else 'Sano')

print("💾 5. Exportando a Excel Maestro...")
# Guardamos todo en un Excel con dos pestañas limpias
with pd.ExcelWriter('Walmart_Dashboard_Master.xlsx') as writer:
    df_master.to_excel(writer, sheet_name='Transacciones_y_Logistica', index=False)
    df_clientes.to_excel(writer, sheet_name='Segmentos_Clientes', index=False)

print("✅ ¡LISTO! Archivo 'Walmart_Dashboard_Master.xlsx' generado con éxito. ¡Descárgalo!")

👨‍💻1. Cargando datos originales...
🧠 2. Recuperando los Clústers de Machine Learning
🔗 3. Fusionando segmentos con la tabla principal de transacciones...
⚙️ 4. Creando KPIs logísticos para el Dashboard...
💾 5. Exportando a Excel Maestro...
✅ ¡LISTO! Archivo 'Walmart_Dashboard_Master.xlsx' generado con éxito. ¡Descárgalo!
